# Graph Zoo Designer
Build, inspect, and save a graph zoo for a simulation run.

**Workflow:**
1. Run the cells for the graph types you want (comment out the rest)
2. Visualize with `zoo.draw_all()`
3. Set a batch name and save
4. Pass the zoo to `ProcessLab`

In [ ]:
from pathlib import Path
from moran_process import GraphZoo, PopulationGraph

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
print(f"Project root: {PROJECT_ROOT}")

## Section 1 — Build the zoo
Run the cells for the graphs you want. Skip the rest.

In [ ]:
BATCH_NAME = "2026-06-03_randomness-4"   # <-- change this for each experiment
DESCRIPTION = "This is just a toy batch to test that random seeding works"  # <-- change this for each experiment
NOTES       = "This is a test batch :) "  # <-- change this for each experiment

R_VALUES   = [1.1]                               # selection coefficients to simulate
N_REPEATS  = 200                              # simulations per graph per r value
N_JOBS = 500
SEED = None


zoo = GraphZoo(name=BATCH_NAME)


In [ ]:
# Mammalian lung (balanced binary tree)
zoo.add(PopulationGraph.mammalian_lung_graph(branching_factor=2, depth=4))

In [ ]:
# Avian lung (parallel rods with inlet/outlet)
zoo.add(PopulationGraph.avian_graph(n_rods=4, rod_length=7))
zoo.add(PopulationGraph.avian_graph(n_rods=7, rod_length=4))


In [ ]:
# Fish gill (comb structure)
zoo.add(PopulationGraph.fish_graph(n_rods=3, rod_length=3))

In [ ]:
# Complete graph (well-mixed baseline)
zoo.add(PopulationGraph.complete_graph(n_nodes=31))

In [ ]:
# Cycle graph
zoo.add(PopulationGraph.cycle_graph(n_nodes=31))

In [ ]:
zoo.add(PopulationGraph.star_graph(n_nodes=31))

In [ ]:
# Random connected graphs (null model) — multiple seeds for statistics
for seed in range(10):
    zoo.add(PopulationGraph.random_connected_graph(n_nodes=31, n_edges=34, seed=seed))

In [ ]:
# Summary
print(zoo)
print(f"Total graphs: {len(zoo)}")

## Section 2 — Visualize

In [ ]:
%matplotlib inline
zoo.draw_all(cols=3)

## Section 3 — Save
This part creates the Batch Directory

In [ ]:
zoo_path = PROJECT_ROOT / "simulation_data" / BATCH_NAME / "zoo.pkl"
zoo.save(str(zoo_path))

## Section 4 — Run simulation
Pass the saved zoo to `ProcessLab`. Use `run_comparative_study` for local runs, `submit_jobs` for HPC.

In [ ]:
from moran_process import GraphZoo, ProcessLab

zoo = GraphZoo.load(str(zoo_path))
lab = ProcessLab()

graph_types = sorted({g.category for g in zoo})
node_sizes  = sorted({g.n_nodes for g in zoo})

# --- Local (small scale) ---
# df = lab.run_comparative_study(
#     graphs_zoo=list(zoo),
#     r_values=R_VALUES,
#     n_repeats=N_REPEATS,
#     output_path=str(PROJECT_ROOT / "simulation_data" / "tmp" / f"batch_{BATCH_NAME}" / "results_local.csv")
# )

# --- HPC (full scale) ---
lab.submit_jobs(
    zoo_path=str(zoo_path),
    r_values=R_VALUES,
    n_repeats=N_REPEATS,
    n_requested_jobs=N_JOBS,
    n_graphs=len(zoo),
    queue="gsla-cpu",
    memory="1GB",
    batch_dir=str(PROJECT_ROOT / "simulation_data" / BATCH_NAME),
    batch_name=BATCH_NAME,
    graph_types=graph_types,
    node_sizes=node_sizes,
    batch_seed=SEED,
    description=DESCRIPTION,
    notes=NOTES,
)